# HyperGrid Real-Time Road Damage Detection (Case Study)
Notebook ini mendemonstrasikan proses inferensi data sensor jalan secara *real-time* menggunakan model **`hg_model_v1.onnx`** dan visualisasi peta spasial GIS.

In [ ]:
import os
import sys
sys.path.insert(0, os.path.abspath('..'))

import numpy as np
import pandas as pd
import folium
from folium.plugins import MarkerCluster, HeatMap
import onnxruntime as rt
from utils.pull_data import pull_road_data
from utils.features import RoadFeatureExtractor

print('Dependencies loaded successfully.')

In [ ]:
# 1. Ambil data dari Supabase (atau fallback ke data sensor lokal jika offline)
data = pull_road_data()

if data and len(data) > 0:
    df_db = pd.DataFrame(data)
    print(f'Berhasil mengambil {len(df_db)} baris data dari Supabase.')
else:
    print('Menggunakan data sensor lokal sebagai simulasi data jalan...')
    fallback_file = '../data/Pothole/trip1_sensors.csv'
    if not os.path.exists(fallback_file):
        fallback_file = './data/Pothole/trip1_sensors.csv'
    df_db = pd.read_csv(fallback_file).head(500)
    print(f'Data simulasi lokal dimuat: {len(df_db)} baris.')

df_db.head()

In [ ]:
# 2. Preprocessing & Ekstraksi 32 Fitur Sensor (Orientation-Invariant)
if 'sensor_x' in df_db.columns:
    df_db['accelerometerX'] = df_db['sensor_x']
    df_db['accelerometerY'] = df_db['sensor_y']
    df_db['accelerometerZ'] = df_db['sensor_z']
if 'speed_kmh' in df_db.columns:
    df_db['speed'] = df_db['speed_kmh']

extractor = RoadFeatureExtractor(window_size=20, step_size=5)
X_stream, metadata = extractor.extract_from_stream(df_db)

print(f'Matriks Fitur Input ONNX: {X_stream.shape}')
X_stream.head()

In [ ]:
# 3. Jalankan Inferensi Model ONNX
model_path = '../models/hg_model_v1.onnx'
if not os.path.exists(model_path):
    model_path = './models/hg_model_v1.onnx'

session = rt.InferenceSession(model_path)
input_name = session.get_inputs()[0].name
label_name = session.get_outputs()[0].name
prob_name = session.get_outputs()[1].name

inputs = X_stream.values.astype(np.float32)
preds, probs = session.run([label_name, prob_name], {input_name: inputs})

pothole_probs = probs[:, 1] if probs.ndim == 2 else probs
metadata['predicted_label'] = preds
metadata['pothole_probability'] = pothole_probs
metadata['status'] = np.where(preds == 1, '⚠️ POTHOLE', '✅ NORMAL')

print('=== HASIL PREDIKSI REAL-TIME ONNX ===')
print(metadata['status'].value_counts())
metadata.head(10)

In [ ]:
# 4. Visualisasi Peta Spasial GIS (Folium)
df_map = metadata.dropna(subset=['latitude', 'longitude']).copy()
center_lat = df_map['latitude'].mean()
center_lon = df_map['longitude'].mean()

m = folium.Map(location=[center_lat, center_lon], zoom_start=15, tiles='OpenStreetMap')
cluster = MarkerCluster(name='Titik Deteksi').add_to(m)

for idx, row in df_map.iterrows():
    is_pothole = row['predicted_label'] == 1
    color = '#e74c3c' if is_pothole else '#3498db'
    title = '⚠️ POTHOLE' if is_pothole else '✅ NORMAL'
    
    popup_content = f"<b>{title}</b><hr><b>Waktu:</b> {row['timestamp']}<br><b>Probabilitas:</b> {row['pothole_probability']:.2%}<br><b>Peak Shock:</b> {row['peak_shock']:.2f} m/s²<br><b>Kecepatan:</b> {row['speed']:.1f} km/h"
    
    folium.CircleMarker(
        location=[row['latitude'], row['longitude']],
        radius=7 if is_pothole else 3,
        color=color,
        fill=True,
        fill_color=color,
        fill_opacity=0.8 if is_pothole else 0.4,
        popup=folium.Popup(popup_content, max_width=250),
    ).add_to(cluster)

# Heatmap untuk Pothole
pothole_points = df_map[df_map['predicted_label'] == 1][['latitude', 'longitude', 'pothole_probability']].values.tolist()
if len(pothole_points) > 0:
    HeatMap(pothole_points, radius=15, blur=10).add_to(m)

folium.LayerControl().add_to(m)
m